# 00 · Build the Quilt-1M impurity table
**Paper:** Section II-A *(Dataset)* · Fig. 1B

QuiltCleaner (Aubreville et al., 2024) released impurity scores for Quilt-1M in two parts:

| Source | Images | Score type |
|---|---:|---|
| Manual audit (`train/val/test_annotations.csv`) | 6,532 | ground truth (0 / 1) |
| ResNet50-D classifier (`predictions_quiltcleaner.csv`) | 646,681 | probability (0 – 1) |

This notebook merges both into **one table covering all 653,213 images**, adds the aggregate
**"Any impairment"** score (the maximum of the eight category scores), and reproduces the
clean / impure statistics of Fig. 1B.

**Output** → `outputs/quilt_impurity_scores.csv`, `outputs/figures/`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from requilt.data import ANY_IMPAIRMENT, IMPURITY_COLUMNS

QUILTCLEANER = P.quiltcleaner_dir
THRESHOLD = cfg.impurity.threshold   # 0.5

## 1 · Manual annotations (ground truth)
The three annotation files are concatenated. They also contain TCGA tiles that are **not** part of
Quilt-1M, so those rows are dropped. Boolean labels become `1.0 / 0.0` so they share the same scale
as the classifier probabilities.

In [ ]:
annotations = pd.concat(
    [pd.read_csv(QUILTCLEANER / f"{split}_annotations.csv") for split in ("train", "val", "test")],
    ignore_index=True,
)
print(f"annotation rows            : {len(annotations):,}")

annotations["Filename"] = annotations["Image"].map(os.path.basename)
annotations = annotations[~annotations["Filename"].str.startswith(cfg.impurity.exclude_prefix)]
print(f"after removing TCGA tiles  : {len(annotations):,}")      # expected: 6,532

annotations[IMPURITY_COLUMNS] = annotations[IMPURITY_COLUMNS].astype(float)
annotations[ANY_IMPAIRMENT] = annotations[IMPURITY_COLUMNS].max(axis=1)

## 2 · Classifier predictions (all remaining images)

In [ ]:
predictions = pd.read_csv(QUILTCLEANER / "predictions_quiltcleaner.csv")
predictions = predictions.loc[:, ~predictions.columns.str.startswith("Unnamed")]  # drop old index
print(f"prediction rows            : {len(predictions):,}")       # expected: 646,681

## 3 · Merge and save
Predictions come first and manual annotations second. Keep this order: later notebooks draw
random samples from this table, and the row order affects which rows a fixed seed selects.

In [ ]:
columns = IMPURITY_COLUMNS + [ANY_IMPAIRMENT, "Filename"]
scores = pd.concat([predictions[columns], annotations[columns]], ignore_index=True)

out_csv = P.output_dir / "quilt_impurity_scores.csv"
scores.to_csv(out_csv, index=False)
print(f"total images               : {len(scores):,}")            # expected: 653,213
print(f"saved -> {out_csv}")

## 4 · Sanity check: every listed image exists on disk

In [ ]:
on_disk = set(os.listdir(P.quilt_images_dir))
missing = scores.loc[~scores["Filename"].isin(on_disk), "Filename"]
print(f"images on disk   : {len(on_disk):,}")
print(f"missing from disk: {len(missing):,}")                       # expected: 0

## 5 · Clean vs. impure statistics
An image is **impure** when its *Any impairment* score is ≥ 0.5.

Expected: **478,228 impure (73.2 %)** and **174,985 clean (26.8 %)**.

In [ ]:
n_total = len(scores)
is_dirty = scores[ANY_IMPAIRMENT] >= THRESHOLD
n_dirty, n_clean = int(is_dirty.sum()), int((~is_dirty).sum())
category_counts = {c: int((scores[c] >= THRESHOLD).sum()) for c in IMPURITY_COLUMNS}

print(f"impure : {n_dirty:>7,}  ({n_dirty / n_total:.1%})")
print(f"clean  : {n_clean:>7,}  ({n_clean / n_total:.1%})")
print("\nper-category prevalence (score >= 0.5):")
for name, count in category_counts.items():
    print(f"  {name:<32} {count:>7,}  ({count / n_total:5.1%})")